In [42]:
import pandas as pd
import os
import random
import numpy as np
from collections import Counter

random.seed(42)
np.random.seed(42)

MODELS = [
    'meta-llama/Llama-3.2-3B-Instruct',
    'Qwen/Qwen2.5-3B-Instruct',
]

NUM_TEST = 2

def add_random_baseline(dataset_df, num=5):
    """
    Genera num predizioni randomiche e fa majority vote.
    """

    # Otteniamo tutte le classi realmente presenti.
    possible_labels = dataset_df['annotations'].apply(
        lambda x: x['irony_appropriateness_label']
    ).unique().tolist()

    #print("Classi disponibili per il random baseline:", possible_labels)

    # Generiamo 5 risposte random
    num += 1
    for i in range(1, num):
        dataset_df[f'random_{i}'] = dataset_df.apply(
            lambda _: random.choice(possible_labels),
            axis=1
        )

    # Majority vote
    def majority_vote(row):
        votes = [
            row[f'random_{i}'] for i in range(1, num)
        ]
        return Counter(votes).most_common(1)[0][0]

    dataset_df['model_answer'] = dataset_df.apply(majority_vote, axis=1)

    return dataset_df

def add_majority_class_baseline(dataset_df):
    majority_label = dataset_df['annotations'].apply(
        lambda x: x['irony_appropriateness_label']
    ).value_counts().idxmax()
    dataset_df['model_answer'] = majority_label
    return dataset_df

def evaluate_aggregated_predictions(dataset_df):
    correct = 0
    total = 0
    for idx, row in dataset_df.iterrows():
        true_label = str(row['annotations']['irony_appropriateness_label'])
        model_answer = str(row[f'model_answer'])
        if model_answer == true_label:
            correct += 1
        total += 1
    accuracy = correct / total if total > 0 else 0

    return accuracy



baseline = False
i = 1
#for _ in range(NUM_TEST):
for model_name in MODELS:
    #dataset_path = f'test{i}_baseline_{model_name.replace("/", "_")}.json'
    dataset_path = f'test2_baseline_{model_name.replace("/", "_")}.json'
    # controllare che il file esista
    if not os.path.exists(dataset_path):
        #print(f'File {dataset_path} does not exist. Skipping evaluation for model {model_name}.')
        continue

    if not baseline:
        random_df = pd.read_json(dataset_path)  # leggere lo stesso dataset
        random_df = add_random_baseline(random_df, num=7)
        result = evaluate_aggregated_predictions(random_df)
        print(f'- Random Baseline Accuracy: {result:.4f}')

        # Baseline 0 fisso
        majority_df = pd.read_json(dataset_path)
        majority_df = add_majority_class_baseline(majority_df)
        majority_acc = evaluate_aggregated_predictions(majority_df)
        print(f'- Zero Baseline Accuracy (always {majority_df["model_answer"].iloc[0]}): {majority_acc:.4f}')

        baseline = True

    dataset_df = pd.read_json(dataset_path)

    result = evaluate_aggregated_predictions(dataset_df)
    print(f'- Test #2 Model: {model_name} - Accuracy: {result:.4f}')
    #print(f'Test #{i} Model: {model_name} - Accuracy: {result:.4f}')
i += 1


- Random Baseline Accuracy: 0.5043
- Zero Baseline Accuracy (always 0): 0.7057
- Test #2 Model: meta-llama/Llama-3.2-3B-Instruct - Accuracy: 0.4729
- Test #2 Model: Qwen/Qwen2.5-3B-Instruct - Accuracy: 0.6152
